## Foundry Agent Tracing with Application Insights

### Installing Libraries and SDKs

In [ ]:
%pip install azure-ai-projects==2.0.0b2 opentelemetry-sdk==1.42.1 azure-core-tracing-opentelemetry==1.0.0b13 azure-monitor-opentelemetry==1.8.8 opentelemetry-exporter-otlp-proto-http==1.42.1

### Importing Libraries and Utilities

In [ ]:
import os

os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import trace
import time
import atexit

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
mcp_server_name = os.getenv("MCP_SERVER_NAME")

### Creating the Foundry Project Client

In [ ]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

In [ ]:
# Test with a lightweight API call
try:
    # Try to get an OpenAI client (this validates the connection)
    test_client = project_client.get_openai_client()
    print("✓ Successfully connected to project and retrieved OpenAI client")
    print(f"Project endpoint: {foundry_project_endpoint}")
except Exception as e:
    print(f"✗ Failed to connect: {type(e).__name__}: {str(e)}")

### Instantiating an OpenAI Client

In [ ]:
openai_client = project_client.get_openai_client()

### Fetching the MCP Server Connection Reference

In [ ]:
connection_id = ""

for connection in project_client.connections.list():
    print(f"Connection Name: {connection.name}")
    if connection.name == mcp_server_name:
        connection_id = connection.id
        break

print(f"The MCP Server Connection ID is: {connection_id}")

### Fetching the App Insights Connection String

In [ ]:
connection_string = project_client.telemetry.get_application_insights_connection_string()
configure_azure_monitor(connection_string=connection_string)

print("App Insights Connection String: {}".format(connection_string))

# Add this: Get the tracer provider to force flush later
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry import trace as trace_api

# Force flush function to ensure traces are sent
def force_flush_traces():
    provider = trace_api.get_tracer_provider()
    if hasattr(provider, 'force_flush'):
        print("Flushing traces to Application Insights...")
        provider.force_flush(timeout_millis=30000)  # 30 second timeout
        time.sleep(2)  # Give some time for the upload

### Beginning the Trace

In [ ]:
tracer = trace.get_tracer(__name__)

with tracer.start_as_current_span("agent-tracing-scenario"):

    tool = MCPTool(
        server_label = "github",
        server_url="https://api.githubcopilot.com/mcp",
        require_approval="never",
        project_connection_id=connection_id
    )

    agent = project_client.agents.create_version(
        agent_name="MCP-My-New-Agent",
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions="You are an intelligent assistant that can interact with the Github MCP server to provide users with relevant repo information and information about all code inside those repositories",
            tools=[tool],
        )
    )

    print(f"Created MCP Agent with ID: {agent.id}")

    # create a conversation to use with the agent
    conversation = openai_client.conversations.create()
    print(f"Created conversation with id: {conversation.id}")

    user_query = "Can you pls help me with the number of repositories I have in my account and list their names as well ?"

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={
            "agent_reference": {
                "name": "MCP-My-New-Agent",
                "type": "agent_reference"
            }
        },
        input=user_query
    )

    print(f"Agent Response: {response.output_text}")


In [ ]:
tracer = trace.get_tracer(__name__)

with tracer.start_as_current_span("agent-tracing-scenario"):

    conversation = openai_client.conversations.create()
    print(f"Created conversation with id: {conversation.id}")

    user_query = "What is my username for github ?"

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={
            "agent_reference": {
                "name": "MCP-My-New-Agent",
                "type": "agent_reference"
            }
        },
        input=user_query
    )

    print(f"Agent Response: {response.output_text}")


In [ ]:
tracer = trace.get_tracer(__name__)

with tracer.start_as_current_span("agent-tracing-scenario"):

    conversation = openai_client.conversations.create()
    print(f"Created conversation with id: {conversation.id}")

    user_query = "Using the tool, can you pls help me with the number of repositories I have in my account and list their names as well ?"

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={
            "agent_reference": {
                "name": "MCP-My-New-Agent",
                "type": "agent_reference"
            }
        },
        input=user_query
    )

    print(f"Agent Response: {response.output_text}")

# IMPORTANT: Force flush traces after each operation
force_flush_traces()

In [ ]:
from opentelemetry.sdk.trace import TracerProvider
provider = trace_api.get_tracer_provider()
print(f"Tracer provider type: {type(provider)}")